# **Vấn đề**

# **Sự mất cân đối giữa tốc độ sản xuất nhựa và hiệu quả quản lý rác thải toàn cầu giai đoạn 1950 - 2019.**

> Thêm khối trích dẫn



**Vấn đề**
1. Trong tổng lượng rác con người thải ra, bao nhiêu phần trăm thực sự được tái chế so với phần bị quản lý kém?
2. Đâu là khu vực đang là điểm nóng gây ra ô nhiễm môi trường
3. Tại sao các quốc gia có thu nhập cao lại có lượng rác quản lý kém trên đầu người thấp hơn các nước đang phát triển, dù họ tiêu thụ nhựa nhiều hơn?
4. Khu vực nào (Châu Á, Châu Phi, Mỹ Latinh...) đang đối mặt với mức độ rác thải nhựa cá nhân cao nhất?"
5. Nếu sản lượng nhựa toàn cầu tiếp tục tăng theo biểu đồ lịch sử, thì lượng rác thải quản lý kém trên đầu người sẽ thay đổi như thế nào nếu hạ tầng không đổi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_folder_path = '/content/drive/MyDrive/CS441V_Data visualization/Final_Project/Data'

In [ ]:
df_prod = pd.read_csv(os.path.join(data_folder_path, '1- global-plastics-production.csv'))
df_ocean = pd.read_csv(os.path.join(data_folder_path, '2- share-of-global-plastic-waste-emitted-to-the-ocean.csv'))
df_fate = pd.read_csv(os.path.join(data_folder_path, '3- share-plastic-fate.csv'))
df_mismanaged = pd.read_csv(os.path.join(data_folder_path, '4- mismanaged-plastic-waste-per-capita.csv'))

# **Làm sạch dữ liệu**

In [ ]:
def clean_data(df):
    # Loại bỏ khoảng trắng thừa trong tên cột
    df.columns = df.columns.str.strip()

    # Kiểm tra và xử lý giá trị thiếu (nếu Code trống thường là các vùng lục địa)
    # Chúng ta sẽ giữ lại nhưng đánh dấu rõ ràng
    df['Code'] = df['Code'].fillna('Region')

    return df

In [ ]:
#Áp dụng hàm làm sạch cơ bản
df_prod = clean_data(df_prod)
df_ocean = clean_data(df_ocean)
df_fate = clean_data(df_fate)
df_mismanaged = clean_data(df_mismanaged)

**Xử lý riêng biệt cho phân tích**

In [ ]:
# Tách dữ liệu Thế giới (World) ra khỏi dữ liệu quốc gia trong file Sản xuất
df_world_prod = df_prod[df_prod['Entity'] == 'World']
df_country_prod = df_prod[df_prod['Entity'] != 'World']

**Đổi tên cột cho ngắn gọn**


In [ ]:
df_fate = df_fate.rename(columns={
    'Share of waste recycled from total regional waste': 'Recycled',
    'Share of waste incinerated from total regional waste': 'Incinerated',
    'Share of littered and mismanaged from total regional waste': 'Mismanaged',
    'Share of waste landfilled from total regional waste': 'Landfilled'
})

df_ocean = df_ocean.rename(columns={
    'Share of global plastics emitted to ocean': 'Ocean_Share'
})

**Hợp nhất dữ liệu (Merge)**

In [ ]:
df_combined_2019 = pd.merge(
    df_ocean[df_ocean['Year'] == 2019],
    df_mismanaged[df_mismanaged['Year'] == 2019],
    on=['Entity', 'Code', 'Year'],
    how='inner'
)

**Lưu dữ liệu đã làm sạch**

In [ ]:
print("Làm sạch hoàn tất!")
print(f"Dữ liệu tổng hợp 2019 có {df_combined_2019.shape[0]} dòng.")
print(df_combined_2019.head())

Làm sạch hoàn tất!
Dữ liệu tổng hợp 2019 có 164 dòng.
                Entity    Code  Year  Ocean_Share  \
0               Africa  Region  2019     7.989317   
1              Albania     ALB  2019     0.159782   
2              Algeria     DZA  2019     0.589510   
3               Angola     AGO  2019     0.087804   
4  Antigua and Barbuda     ATG  2019     0.000204   

   Mismanaged plastic waste per capita (kg per year)  
0                                          10.465928  
1                                          24.239153  
2                                          17.758995  
3                                           7.445279  
4                                           6.463918  


**Kiểm tra dữ dữ liệu đã làm sạch**

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Giả sử các file đã được load
dfs = {'Production': df_prod, 'Ocean': df_ocean, 'Fate': df_fate, 'PerCapita': df_mismanaged}

for name, df in dfs.items():
    print(f"--- {name} ---")
    # Kiểm tra thiếu dữ liệu và kiểu dữ liệu
    print(df.info())
    # Kiểm tra các dòng đại diện cho khu vực (Region) thay vì quốc gia
    print("Các khu vực tiêu biểu:", df[df['Code'].isna()]['Entity'].unique()[:5])

--- Production ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 4 columns):
 #   Column                                           Non-Null Count  Dtype 
---  ------                                           --------------  ----- 
 0   Entity                                           69 non-null     object
 1   Code                                             69 non-null     object
 2   Year                                             69 non-null     int64 
 3   Annual plastic production between 1950 and 2019  69 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 2.3+ KB
None
Các khu vực tiêu biểu: []
--- Ocean ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Entity       170 non-null    object 
 1   Code         170 non-null    object 
 2   Year         170 non-null    int64  
 3   Oce

# **Phân tích & Trực quan hóa**

#**Vấn đề 1 : Tỉ lệ Tái chế so với quản lý kém**


In [ ]:
df_prod_world = df_prod[(df_prod['Entity'] == 'World') & (df_prod['Year'] >= 2000)].copy()
df_fate_world = df_fate[(df_fate['Entity'] == 'World') & (df_fate['Year'] >= 2000)].copy()

df_merge = pd.merge(df_prod_world[['Year', 'Annual plastic production between 1950 and 2019']],
                    df_fate_world[['Year', 'Recycled']],
                    on='Year')

# Khối lượng tái chế thực tế = Tổng sản lượng * (Tỷ lệ phần trăm / 100)
df_merge['Recycled_Tonnes'] = (df_merge['Annual plastic production between 1950 and 2019'] * df_merge['Recycled'] / 100)
fig = go.Figure()

# Đường Tổng sản lượng nhựa
fig.add_trace(go.Scatter(
    x=df_merge['Year'],
    y=df_merge['Annual plastic production between 1950 and 2019'],
    name="Tổng sản lượng sản xuất (M tấn)",
    line=dict(color='firebrick', width=4)
))

# Đường Khối lượng thực tế được tái chế (đã quy đổi)
fig.add_trace(go.Scatter(
    x=df_merge['Year'],
    y=df_merge['Recycled_Tonnes'],
    name="Khối lượng thực tế tái chế (M tấn)",
    fill='tozeroy', # Tô màu vùng dưới đường tổng sản lượng
    line=dict(color='royalblue', width=4)
))

# Cấu hình Layout
fig.update_layout(
    title='Khối lượng sản xuất tăng vọt so với nỗ lực tái chế (Đơn vị:M tấn)',
    xaxis_title='Năm',
    yaxis_title='Khối lượng nhựa (M tấn)',
    template='plotly_white',
    height=700,
    legend=dict(x=0, y=1.1, orientation='h')
)

fig.show()

# In kết quả
latest = df_merge.iloc[-1]
first = df_merge.iloc[0]

print(f"Năm 2000: Sản xuất {first['Annual plastic production between 1950 and 2019']:,.0f} tấn -> Tái chế được: {first['Recycled_Tonnes']:,.0f} M tấn")
print(f"Năm 2019: Sản xuất {latest['Annual plastic production between 1950 and 2019']:,.0f} tấn -> Tái chế được: {latest['Recycled_Tonnes']:,.0f} M tấn")
print(f"-----------------------------------")
print(f"Dù tỷ lệ tái chế tăng, nhưng lượng nhựa 'BỊ BỎ LẠI' (không tái chế) "
      f"đã tăng từ {first['Annual plastic production between 1950 and 2019'] - first['Recycled_Tonnes']:,.0f} triệu tấn "
      f"lên đến {latest['Annual plastic production between 1950 and 2019'] - latest['Recycled_Tonnes']:,.0f} triệu tấn.")

Năm 2000: Sản xuất 213,000,000 tấn -> Tái chế được: 7,888,213 M tấn
Năm 2019: Sản xuất 459,746,020 tấn -> Tái chế được: 42,721,182 M tấn
-----------------------------------
Dù tỷ lệ tái chế tăng, nhưng lượng nhựa 'BỊ BỎ LẠI' (không tái chế) đã tăng từ 205,111,787 triệu tấn lên đến 417,024,838 triệu tấn.


 Trong tổng lượng rác thải ra, bao nhiêu % thực sự được tái chế so với phần bị quản lý kém?

Mặc dù công nghệ tái chế được quảng bá rộng rãi, nhưng con số thực tế rất khiêm tốn. Tỉ lệ tái chế toàn cầu chỉ nhích dần từ khoảng 3.7% (năm 2000) lên gần 9% (năm 2019). Trong khi đó, lượng rác quản lý kém và xả thải trực tiếp luôn duy trì ở mức cao gấp 2-3 lần lượng tái chế.



**Kết luận:** Hệ thống tái chế hiện tại đang thất bại trong việc đuổi kịp quy mô tiêu thụ. Chúng ta đang tập trung vào một giải pháp (tái chế) chỉ xử lý được chưa đầy 1/10 vấn đề.

# **Vấn đề 2: Điểm nóng ô nhiễm môi trường**

In [ ]:
import plotly.express as px

# Lọc dữ liệu sạch
df_ocean_clean = df_ocean[df_ocean['Code'].notna()]

#TỔNG LƯỢNG RÁC RA ĐẠI DƯƠNG (%) ---
fig_a = px.choropleth(
    df_ocean_clean,
    locations="Code",
    color="Ocean_Share",
    hover_name="Entity",
    title="Các điểm nóng xả rác ra đại dương (% toàn cầu)",
    # Sử dụng dải màu từ Vàng -> Cam -> Đỏ rực
    color_continuous_scale=["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"],
    # GIỚI HẠN MÀU: Vì đa số các nước < 10%
    range_color=[0, 50],
    template='plotly_white'
)

# Cấu hình
fig_a.update_layout(
    autosize=True,
    height=850,
    margin=dict(l=0, r=0, t=60, b=0),
    title_x=0.5,
    title_font=dict(size=24, color='#2c3e50'),
    coloraxis_colorbar=dict(
        title="Tỉ lệ %",
        thickness=20,
        len=0.5,
        ticksuffix="%"
    )
)

fig_a.update_geos(
    projection_type="natural earth",
    showcountries=True,
    countrycolor="#dcdde1",
    showocean=True,
    oceancolor="#f5f6fa"
)

fig_a.show()

Dữ liệu chỉ ra một sự tập trung cực đoan: Hơn 80% lượng rác nhựa trôi ra đại dương đến từ các con sông ở Châu Á. Philippines là "điểm nóng" lớn nhất toàn cầu (chiếm hơn 36%).

**Kết luận:** Ô nhiễm đại dương là vấn đề địa lý đặc thù. Rác thải nhựa trôi ra biển do các nước này tiêu thụ kết hợp giữa mật độ dân số ven sông cao và địa hình sông ngòi đóng vai trò như đường dẫn rác trực tiếp ra biển.

# **Vấn đề 3: Nghịch lý giữa Nước phát triển với Nước đang phát triển**

In [ ]:
countries = ['United States', 'United Kingdom', 'Germany', 'Vietnam', 'Philippines', 'India']
df_compare = df_mismanaged[df_mismanaged['Entity'].isin(countries)]

fig3 = px.bar(df_compare, x='Entity', y='Mismanaged plastic waste per capita (kg per year)', color='Entity',
             title='So sánh giữ các nước phát triển và đang phát triển về mức độ rác thải',
             labels={'Mismanaged plastic waste per capita (kg per year)': 'Rác thải nhựa(kg/người/năm)'})
fig3.show()

 Dữ liệu cho thấy một thực trạng tương phản: Người dân tại các quốc gia thu nhập cao (Mỹ, Châu Âu) tiêu thụ lượng nhựa gấp nhiều lần so với các nước nghèo, nhưng chỉ số rác thải "quản lý kém" (mismanaged) của họ chưa vượt ngưỡng 2. Điều này không phải do họ xả rác ít đi, mà vì họ có hệ thống thu gom và xử lý khép kín cực kỳ hiệu quả.

**Kết luận:** Quản lý rác thải kém không tỷ lệ thuận với lượng tiêu thụ mà tỷ lệ thuận với năng lực kinh tế. Ô nhiễm nhựa ở các nước đang phát triển không phải do ý thức người dân kém hơn, mà do họ đang sống trong những vùng "trống" về dịch vụ hạ tầng môi trường.

#**Vấn đề 4: Khu vực có mức rác thải nhựa cá nhân cao nhất**

In [ ]:
import plotly.express as px

# Lọc dữ liệu sạch
df_per_capita_clean = df_mismanaged[df_mismanaged['Code'].notna()]

# ---RÁC QUẢN LÝ KÉM BÌNH QUÂN ĐẦU NGƯỜI (kg/năm) ---
fig_b = px.choropleth(
    df_per_capita_clean,
    locations="Code",
    color="Mismanaged plastic waste per capita (kg per year)", # Tên cột gốc
    hover_name="Entity",
    title="Khối lượng rác thải do quản lý kém tính trên bình quân đầu người (kg/năm)",
    color_continuous_scale=px.colors.sequential.YlOrBr,
    range_color=[0, 25],
    template='plotly_white'
)

# Cấu hình bao trọn màn hình và tràn viền tương tự Map A
fig_b.update_layout(
    autosize=True,
    height=850,
    margin=dict(l=0, r=0, t=60, b=0),
    title_x=0.5,
    title_font=dict(size=24, color='#80391e'), # Màu tiêu đề tông nâu đỏ cho hợp bản đồ
    coloraxis_colorbar=dict(
        title="kg/người",
        thickness=20,
        len=0.5,
        ticksuffix=" kg"
    )
)

fig_b.update_geos(
    projection_type="natural earth",
    showcountries=True,
    countrycolor="#dcdde1",
    showocean=True,
    oceancolor="#eef2f7"
)

fig_b.show()

Gánh nặng thực tế: Khi phân tích chỉ số rác thải quản lý kém trên đầu người (kg/người), bản đồ hiển thị màu sắc đậm nhất tại các khu vực Châu Phi, Nam Á và các Đảo quốc nhỏ. Tại những nơi này, trung bình một cá nhân phải đối mặt với hơn 20kg rác nhựa bị thải trực tiếp ra môi trường mỗi năm mà không có ai thu gom.

**Kết luận:** Các quốc gia đang phát triển và hải đảo là những nơi đang gánh chịu hậu quả trực tiếp nhất của rác thải nhựa. Đây là những khu vực trọng điểm cần được ưu tiên hỗ trợ về công nghệ và hạ tầng xử lý rác từ cộng đồng quốc tế để ngăn chặn thảm họa môi trường tại chỗ.

#**Vấn đề 5: Dự đoán rác thải nhựa đến năm 2030**






In [ ]:
# Tính tốc độ tăng trưởng bình quân mỗi năm (triệu tấn/năm) từ 2000-2019
years_hist = df_merge['Year'].values
prod_hist = df_merge['Annual plastic production between 1950 and 2019'].values
recy_hist = df_merge['Recycled_Tonnes'].values

# Dùng hồi quy tuyến tính đơn giản để tìm xu hướng tăng trưởng
slope_prod = np.polyfit(years_hist, prod_hist, 1)[0]
slope_recy = np.polyfit(years_hist, recy_hist, 1)[0]

# --- DỰ BÁO ĐẾN NĂM 2030 ---
years_future = np.arange(2020, 2031)
prod_future = prod_hist[-1] + slope_prod * (years_future - 2019)
recy_future = recy_hist[-1] + slope_recy * (years_future - 2019)

# ---TRỰC QUAN HÓA ---
fig = go.Figure()

# Vẽ dữ liệu lịch sử
fig.add_trace(go.Scatter(x=years_hist, y=prod_hist, name="Sản lượng thực tế (2000-2019)", line=dict(color='firebrick', width=3)))
fig.add_trace(go.Scatter(x=years_hist, y=recy_hist, name="Tái chế thực tế (2000-2019)", line=dict(color='royalblue', width=3)))

# Vẽ dữ liệu dự báo (Dùng đường đứt nét - Dashed line)
fig.add_trace(go.Scatter(x=years_future, y=prod_future, name="Dự đoán Sản lượng 2030", line=dict(color='firebrick', width=3, dash='dash')))
fig.add_trace(go.Scatter(x=years_future, y=recy_future, name="Dự đoán Tái chế 2030", line=dict(color='royalblue', width=3, dash='dash')))

# Thêm ghi chú cho điểm năm 2030
fig.add_annotation(x=2030, y=prod_future[-1], text=f"2030: ~{prod_future[-1]/1e6:.1f}M Tấn", showarrow=True, arrowhead=1)
fig.add_annotation(x=2030, y=recy_future[-1], text=f"2030: ~{recy_future[-1]/1e6:.1f}M Tấn", showarrow=True, arrowhead=1)

fig.update_layout(
    title='DỰ BÁO 2030: KHOẢNG CÁCH KHỔNG LỒ GIỮA RÁC THẢI VÀ TÁI CHẾ',
    xaxis_title='Năm',
    yaxis_title='Khối lượng nhựa (Tấn)',
    template='plotly_white',
    height=700
)

fig.show()

# --- 4. KẾT LUẬN CON SỐ ---
print(f"DỰ BÁO ĐẾN NĂM 2030:")
print(f"- Sản lượng nhựa sản xuất: ~{prod_future[-1]/1e6:.1f} triệu tấn")
print(f"- Lượng nhựa được tái chế: ~{recy_future[-1]/1e6:.1f} triệu tấn")
print(f"- Lượng nhựa QUẢN LÝ KÉM (không được xử lý): ~{(prod_future[-1] - recy_future[-1])/1e6:.1f} triệu tấn")

DỰ BÁO ĐẾN NĂM 2030:
- Sản lượng nhựa sản xuất: ~596.8 triệu tấn
- Lượng nhựa được tái chế: ~62.3 triệu tấn
- Lượng nhựa QUẢN LÝ KÉM (không được xử lý): ~534.5 triệu tấn


Dựa trên mô hình dự báo tăng trưởng 4.1%/năm, sản lượng nhựa năm 2030 sẽ chạm mốc khổng lồ ~600 triệu tấn. Nếu hạ tầng xử lý không được cải thiện đột phá, lượng rác quản lý kém sẽ tăng thêm khoảng 140 triệu tấn/năm.


**Kết luận:** Nếu không kiểm soát sản lượng nhựa nguyên sinh ngay từ đầu nguồn, mọi nỗ lực cải thiện hạ tầng sẽ chỉ là cuộc rượt đuổi vô tận và vô vọng.

# **So sánh với bài giữa kỳ**

# **Nội dung giải thích sự cải thiện từ Midterm đến Final Project**

1. Sự thống nhất và tính liên kết của vấn đề:
Ở bài giữa kỳ, chúng em phân tích hai chủ đề riêng biệt (Mực nước biển & Tai nạn giao thông), dẫn đến sự thiếu mạch lạc trong câu chuyện dữ liệu. Với dự án này, chúng em tập trung vào một hệ sinh thái dữ liệu duy nhất nhưng đa chiều: Ô nhiễm nhựa. Nhóm đã kết nối thành công quy trình từ khâu Sản xuất đến Quản lý xử lý và cuối cùng là Hậu quả môi trường đại dương, tạo ra một chuỗi Storytelling hoàn chỉnh và logic.

2. Độ phức tạp trong xử lý và khai phá dữ liệu:
Thay vì chỉ sử dụng các bộ dữ liệu đơn lẻ có sẵn, bài cuối kỳ thực hiện kỹ thuật Data Merging (Hợp nhất) từ 4 tệp dữ liệu lớn có mối liên hệ mật thiết. Chúng em đã tự xây dựng các bộ quy đổi chỉ số mới (từ tỉ lệ % sang khối lượng Tấn thực tế) để bóc tách sự chênh lệch giữa tốc độ sản xuất và năng lực tái chế. Đồng thời, việc đối chiếu giữa chỉ số tổng thể và chỉ số bình quân đầu người đã giúp tìm ra những "điểm nóng" ẩn mình mà cách phân tích thông thường không thể thấy được.

3. Kỹ thuật trực quan hóa và Phân tích dự báo:
Nhóm đã nâng cấp từ các biểu đồ tĩnh (Bar/Pie) đơn giản ở bài Midterm sang các Interactive Dashboard (Bản đồ tương tác Choropleth và biểu đồ đa trục). Chúng em không chỉ dừng lại ở việc mô tả dữ liệu quá khứ mà còn thực hiện Dự báo xu hướng (Forecasting) đến năm 2030 dựa trên tốc độ tăng trưởng lịch sử. Điều này giúp bài phân tích có tính ứng dụng cao khi chỉ ra được sự hụt hơi của hạ tầng trước tốc độ sản xuất nhựa toàn cầu.

Kết luận:
Thay vì chỉ liệt kê dữ liệu "tăng hay giảm", chúng em đã sử dụng dữ liệu để giải mã các nghịch lý về quản lý rác thải giữa các nhóm quốc gia. Từ đó, dự án đưa ra các kết luận sắc bén về việc ưu tiên can thiệp hạ tầng tại các khu vực trọng điểm, thể hiện tư duy phân tích phản biện và kỹ năng giải quyết vấn đề bằng dữ liệu.